In [4]:
from pathlib import Path
import sys

# Add src to path
sys.path.append('../src_new')

In [5]:
import numpy as np
from dem import MarsDEM   # change this to the filename containing your class


DEM_PATH = "../data/Mars_MGS_MOLA_DEM_mosaic_global_463m.tif"


def main():

    dem = MarsDEM(DEM_PATH)

    try:
        # ---------------------------------------------------------
        # 1. Basic DEM information
        # ---------------------------------------------------------
        print("========== DEM INFO ==========")
        print("Width:       ", dem.width)
        print("Height:      ", dem.height)
        print("NoData:      ", dem.nodata)
        print("Transform:   ", dem.transform)
        print("CRS:         ", dem.src.crs)
        print("Bounds:      ", dem.src.bounds)
        print()

        # ---------------------------------------------------------
        # 2. Test one coordinate
        # ---------------------------------------------------------
        lat = -14.5
        lon = 42.3

        print("========== SINGLE POINT ==========")
        print(f"Input latitude:  {lat}")
        print(f"Input longitude: {lon}")

        row, col = dem.lonlat_to_pixel(lat, lon)

        print(f"Pixel row: {row}")
        print(f"Pixel col: {col}")

        elevation = dem.get_elevation(lat, lon)

        print(f"Interpolated elevation: {elevation} m")
        print()

        # ---------------------------------------------------------
        # 3. Test multiple coordinates
        # ---------------------------------------------------------
        print("========== MULTIPLE POINTS ==========")

        lats = np.array([
            -14.5,
            -15.0,
            -16.0,
            0.0,
            20.0
        ])

        lons = np.array([
            42.3,
            43.0,
            44.0,
            0.0,
            100.0
        ])

        elevations = dem.get_elevation(lats, lons)

        for lat, lon, elevation in zip(lats, lons, elevations):
            print(
                f"Lat: {lat:7.2f}, "
                f"Lon: {lon:7.2f}, "
                f"Elevation: {elevation:10.2f} m"
            )

        print()

        # ---------------------------------------------------------
        # 4. Test longitude wrapping
        # ---------------------------------------------------------
        print("========== LONGITUDE WRAPPING ==========")

        lat = -14.5

        lon1 = 270.0
        lon2 = -90.0

        row1, col1 = dem.lonlat_to_pixel(lat, lon1)
        row2, col2 = dem.lonlat_to_pixel(lat, lon2)

        elev1 = dem.get_elevation(lat, lon1)
        elev2 = dem.get_elevation(lat, lon2)

        print(f"{lon1}° → row={row1}, col={col1}")
        print(f"{lon2}° → row={row2}, col={col2}")

        print(f"Elevation at {lon1}°: {elev1} m")
        print(f"Elevation at {lon2}°: {elev2} m")

        print("Same pixel:",
              np.isclose(row1, row2) and np.isclose(col1, col2))

        print("Same elevation:",
              np.isclose(elev1, elev2, equal_nan=True))

        print()

        # ---------------------------------------------------------
        # 5. Compare interpolated value with nearest pixel
        # ---------------------------------------------------------
        print("========== INTERPOLATION TEST ==========")

        lat = -14.5
        lon = 42.3

        row, col = dem.lonlat_to_pixel(lat, lon)

        interpolated = dem.get_elevation(lat, lon)

        nearest_row = int(round(float(row)))
        nearest_col = int(round(float(col)))

        if (
            0 <= nearest_row < dem.height
            and 0 <= nearest_col < dem.width
        ):
            nearest = dem.band[nearest_row, nearest_col]

            print(f"Pixel position:      ({row:.4f}, {col:.4f})")
            print(f"Nearest pixel:       ({nearest_row}, {nearest_col})")
            print(f"Nearest elevation:   {nearest} m")
            print(f"Bilinear elevation:  {interpolated:.2f} m")

        print()

        # ---------------------------------------------------------
        # 6. Test __call__()
        # ---------------------------------------------------------
        print("========== __call__ TEST ==========")

        elevation1 = dem.get_elevation(-14.5, 42.3)
        elevation2 = dem(-14.5, 42.3)

        print("get_elevation():", elevation1)
        print("__call__():     ", elevation2)
        print("Equal:",
              np.isclose(elevation1, elevation2, equal_nan=True))

    finally:
        dem.close()


if __name__ == "__main__":
    main()

========== DEM INFO ==========
Width:        46080
Height:       23040
NoData:       -32768.0
Transform:    | 463.09, 0.00,-10669675.20|
| 0.00,-463.09, 5334837.60|
| 0.00, 0.00, 1.00|
CRS:          PROJCS["Equirectangular Mars",GEOGCS["GCS_Mars",DATUM["D_Mars",SPHEROID["Mars_localRadius",3396190,0]],PRIMEM["Reference_Meridian",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Equirectangular"],PARAMETER["standard_parallel_1",0],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Bounds:       BoundingBox(left=-10669675.197320545, bottom=-5334837.598660273, right=10669675.197320545, top=5334837.598660273)

========== SINGLE POINT ==========
Input latitude:  -14.5
Input longitude: 42.3
Pixel row: 13375.960053363115
Pixel col: 28454.283466017914
Interpolated elevation: 2938.0 m

========== MULTIPLE POINTS ==========
Lat:  -14.50, Lon:   42.